# Project 03 — House Price Prediction

**Difficulty:** Beginner  
**Skills:** Regression, Feature Engineering, scikit-learn Pipelines  
**Dataset:** California Housing (sklearn built-in — ~20k samples)

## Objective
Predict median house values in California census districts from demographic and geographic features. A classic regression problem.

## Pipeline
1. Load & explore data
2. Visualise feature relationships
3. Feature engineering
4. Train and evaluate regression models
5. Analyse residuals and errors

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

## 1. Load & Explore

In [ ]:
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target  # median house value in $100k

print('Shape:', df.shape)
print('Target unit: $100,000s (so 2.0 = $200,000)')
df.head()

In [ ]:
print(df.describe().T.round(3))
print('\nMissing values:', df.isna().sum().sum())

In [ ]:
# Feature descriptions
feature_descriptions = {
    'MedInc':    'Median income (in $10k)',
    'HouseAge':  'Median house age (years)',
    'AveRooms':  'Average rooms per household',
    'AveBedrms': 'Average bedrooms per household',
    'Population':'Block population',
    'AveOccup':  'Average household size',
    'Latitude':  'Latitude',
    'Longitude': 'Longitude'
}
for k, v in feature_descriptions.items():
    print(f'{k:12s}: {v}')

## 2. Exploratory Analysis

In [ ]:
# Target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df['MedHouseVal'], bins=50, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('House Value Distribution')
axes[0].set_xlabel('Median House Value ($100k)')
axes[0].axvline(df['MedHouseVal'].mean(), color='red', linestyle='--',
                label=f'Mean: ${df["MedHouseVal"].mean()*100:.0f}k')
axes[0].axvline(df['MedHouseVal'].median(), color='green', linestyle='--',
                label=f'Median: ${df["MedHouseVal"].median()*100:.0f}k')
axes[0].legend()

# Log transform check
sns.histplot(np.log1p(df['MedHouseVal']), bins=50, kde=True, ax=axes[1], color='coral')
axes[1].set_title('Log-transformed House Value')
axes[1].set_xlabel('log(Median House Value + 1)')

plt.suptitle('Target Variable: Median House Value', fontsize=13)
plt.tight_layout()
plt.show()
print(f'Max capped at $500k (value=5.0): {(df["MedHouseVal"]==5.0).sum()} rows')

In [ ]:
# Correlation with target
corr = df.corr()['MedHouseVal'].sort_values(ascending=False)
print('Correlation with House Value:')
print(corr.round(3))

In [ ]:
# Geographic price map
fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(df['Longitude'], df['Latitude'],
                     c=df['MedHouseVal'], cmap='RdYlGn', s=1, alpha=0.5)
plt.colorbar(scatter, ax=ax, label='Median House Value ($100k)')
ax.set_title('California House Values by Location', fontsize=13)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
plt.tight_layout()
plt.show()

In [ ]:
# Key relationship: Income vs Price
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Sample to avoid slow scatter render
sample = df.sample(2000, random_state=42)
sns.scatterplot(data=sample, x='MedInc', y='MedHouseVal',
                alpha=0.3, color='steelblue', ax=axes[0])
axes[0].set_title('Median Income vs House Value')
axes[0].set_xlabel('Median Income ($10k)')
axes[0].set_ylabel('Median House Value ($100k)')

sns.scatterplot(data=sample, x='HouseAge', y='MedHouseVal',
                alpha=0.3, color='coral', ax=axes[1])
axes[1].set_title('House Age vs Value')
axes[1].set_xlabel('House Age (years)')

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
df_feat = df.copy()

# Rooms per person (normalise rooms by household size)
df_feat['rooms_per_person']    = df_feat['AveRooms']   / df_feat['AveOccup']
df_feat['bedrooms_per_room']   = df_feat['AveBedrms']  / df_feat['AveRooms']
df_feat['population_per_household'] = df_feat['Population'] / df_feat['AveOccup']

# Income buckets
df_feat['income_bracket'] = pd.cut(df_feat['MedInc'],
                                    bins=[0, 1.5, 3.0, 4.5, 6.0, 15],
                                    labels=['very_low','low','medium','high','very_high'])

# Remove outliers in rooms_per_person
df_feat = df_feat[df_feat['rooms_per_person'] < 20]
df_feat = df_feat[df_feat['bedrooms_per_room'] < 1]

print('Shape after feature engineering:', df_feat.shape)
new_features = ['rooms_per_person', 'bedrooms_per_room', 'population_per_household']
new_corr = df_feat[new_features + ['MedHouseVal']].corr()['MedHouseVal'].drop('MedHouseVal')
print('\nNew feature correlations with target:')
print(new_corr.round(3))

## 4. Train & Evaluate Models

In [ ]:
feature_cols = ['MedInc','HouseAge','AveRooms','AveBedrms','Population',
                'AveOccup','Latitude','Longitude',
                'rooms_per_person','bedrooms_per_room','population_per_household']

X = df_feat[feature_cols].values
y = df_feat['MedHouseVal'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

In [ ]:
models = {
    'Linear Regression':    Pipeline([('scaler', StandardScaler()), ('reg', LinearRegression())]),
    'Ridge (α=1.0)':        Pipeline([('scaler', StandardScaler()), ('reg', Ridge(alpha=1.0))]),
    'Lasso (α=0.01)':       Pipeline([('scaler', StandardScaler()), ('reg', Lasso(alpha=0.01))]),
    'Random Forest':        RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':    GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
}

def evaluate(name, model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    mae  = mean_absolute_error(y_te, pred)
    r2   = r2_score(y_te, pred)
    return {'Model': name, 'RMSE ($100k)': rmse, 'MAE ($100k)': mae, 'R²': r2}

results = [evaluate(n, m, X_train, y_train, X_test, y_test) for n, m in models.items()]
results_df = pd.DataFrame(results).sort_values('R²', ascending=False)
print(results_df.round(4).to_string(index=False))

In [ ]:
# Visualise model comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

results_df.plot.barh('Model', 'R²', ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('R² Score by Model (higher = better)')
axes[0].set_xlim(0, 1)
for bar in axes[0].patches:
    axes[0].text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                 f'{bar.get_width():.3f}', va='center')

results_df.plot.barh('Model', 'RMSE ($100k)', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('RMSE by Model (lower = better)')
for bar in axes[1].patches:
    axes[1].text(bar.get_width()+0.01, bar.get_y()+bar.get_height()/2,
                 f'${bar.get_width()*100:.0f}k', va='center')

plt.suptitle('Model Comparison — House Price Prediction', fontsize=13)
plt.tight_layout()
plt.show()

## 5. Best Model — Residual Analysis

In [ ]:
best = models['Gradient Boosting']
y_pred = best.predict(X_test)
residuals = y_test - y_pred

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Predicted vs Actual
axes[0,0].scatter(y_test, y_pred, alpha=0.2, color='steelblue', s=5)
axes[0,0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0,0].set_title('Predicted vs Actual')
axes[0,0].set_xlabel('Actual ($100k)')
axes[0,0].set_ylabel('Predicted ($100k)')

# Residual distribution
axes[0,1].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[0,1].axvline(0, color='red', linestyle='--')
axes[0,1].set_title('Residual Distribution')
axes[0,1].set_xlabel('Residual ($100k)')

# Residuals vs Predicted
axes[1,0].scatter(y_pred, residuals, alpha=0.2, color='steelblue', s=5)
axes[1,0].axhline(0, color='red', linestyle='--')
axes[1,0].set_title('Residuals vs Predicted')
axes[1,0].set_xlabel('Predicted ($100k)')
axes[1,0].set_ylabel('Residual')

# Feature importances
gb = best
feat_imp = pd.Series(gb.feature_importances_, index=feature_cols).sort_values(ascending=True)
feat_imp.plot.barh(color='steelblue', ax=axes[1,1], edgecolor='white')
axes[1,1].set_title('Feature Importances')
axes[1,1].set_xlabel('Importance')

plt.suptitle('Gradient Boosting — Model Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f'Best model: Gradient Boosting')
print(f'R² = {r2:.3f}  (explains {r2:.1%} of variance in house prices)')
print(f'RMSE = ${rmse*100:.0f}k  (typical prediction error)')

## Key Findings

| Finding | Detail |
|---------|--------|
| **Best predictor** | `MedInc` — by far the strongest single feature |
| **Location matters** | `Latitude`/`Longitude` together capture coastal premium |
| **Feature engineering helped** | `rooms_per_person` correlates better than raw rooms |
| **Best model** | Gradient Boosting (R² ≈ 0.83, RMSE ≈ $50k) |
| **Residuals** | Right-skewed — model underpredicts expensive homes (capped at $500k) |

## Next Steps
- Try XGBoost / LightGBM — often 2-5% better
- Tune hyperparameters with `GridSearchCV` or `Optuna`
- Address the $500k price cap with log-transform of target
- Add interaction features (income × latitude)